In [ ]:
import polars as pl
import numpy as np

In [ ]:
df = pl.read_parquet('prices.parquet')

In [ ]:
df = df.with_columns(
    pl.col('date').dt.year().alias('year'),
    pl.col('date').dt.month().alias('month')
)

In [ ]:
df.filter([
    pl.col('ticker') =='LUV',
]).sort('date', descending=True)

In [ ]:
monthly_prices = (
    df.sort(['ticker', 'date'])
    .group_by(['ticker', 'year', 'month'])
    .agg(pl.col('close').last().alias('price'))
    .sort(['ticker', 'year', 'month'])
)

In [ ]:
monthly_returns = monthly_prices.with_columns(
    (pl.col('price') / pl.col('price').shift(1) - 1)
    .over('ticker')
    .alias('return')
)

In [ ]:
monthly_returns = monthly_returns.with_columns(
    (pl.col('year').cast(pl.String) + '-' + 
     pl.col('month').cast(pl.String).str.zfill(2))
    .alias('period')
)

In [ ]:
wide = monthly_returns.pivot(
    values='return',
    index='period',
    on='ticker'
).sort('period')

In [ ]:
periods = wide.select('period')

In [ ]:
# Compute formation returns on each stock
formation = wide.select(
    pl.col(col)
    .shift(2)
    .rolling_map(lambda x: (1 + x).product() - 1, window_size=11)
    .alias('{}_formulation'.format(col))

    for col in wide.columns if col != 'period'
)

In [ ]:
formation = periods.hstack(formation)

In [ ]:
cols = formation.columns[1:]

In [ ]:
mean = formation.select(cols).unpivot().select(pl.col('value').mean())[0,0]

print(f"Mean formation return: {mean:.4f}")

In [ ]:
# Convert formation data into long format with month as index
formation_long = formation.unpivot(
    index='period',
    variable_name='ticker',
    value_name='formation_return'
).drop_nulls()

In [ ]:
# Fix ticker names and compute ranks grouped by period
formation_long = formation_long.with_columns(
    pl.col('ticker').str.replace('_formulation', ''),
    pl.col('formation_return').rank()
    .over('period').alias('rank')
)

In [ ]:
# Tuen ranks into deciles using ceil and / 10
formation_long = formation_long.with_columns(
    (pl.col('rank') / 10).ceil().cast(pl.Int32).alias('decile')
)

In [ ]:
# Convert returns into the same month
returns_long = wide.unpivot(
    index='period',
    variable_name='ticker',
    value_name='return'
)

In [ ]:
# Join our formation calculations into the long returns df. Joined on period and ticker
joined_df = returns_long.join(formation_long, on=['period', 'ticker'], how='inner').drop('rank')

In [ ]:
# Calculate decile returns
decile_returns = joined_df.group_by(['period', 'decile']).agg([
    pl.col('return').mean().alias('decile_return')
]).sort(['period', 'decile'])

In [ ]:
decile_returns

In [ ]:
# Costruct WML portfolio by taking difference between decile 51 and 1
wml = decile_returns.filter(pl.col('decile') == 51).join(
    decile_returns.filter(pl.col('decile') == 1),
    on='period',
    suffix='_loser'
).with_columns(
    (pl.col('decile_return') - pl.col('decile_return_loser')).alias('wml_return')
).select(['period', 'wml_return'])

In [ ]:
import scipy.stats as stats

wml_vals = wml['wml_return'].to_numpy()

mean = wml_vals.mean()
sharpe = (mean / wml_vals.std()) * np.sqrt(12)
skewness = stats.skew(wml_vals)

print(f"Mean monthly return: {mean:.4f}")
print(f"Annualised Sharpe: {sharpe:.4f}")
print(f"Skewness: {skewness:.4f}")

# Worst 5 months
print("\nWorst 5 WML months:")
print(wml.sort('wml_return').head(5))

In [ ]:
# Skewness is positive (+0.53) contrary to the paper's finding (-4.70).
# Likely explanations:
# 1. Large cap universe — loser stocks don't accumulate extreme beta
# 2. Sample period (2011-2026) includes strong bull runs in 2023-2024
#    where winner stocks (tech/AI) dramatically outperformed losers
# 3. The paper's negative skewness is driven by pre-WWII crashes not 
#    present in our sample
# Gate check deferred — qualitative momentum premium confirmed (+1.11%/month)
# but skewness finding diverges from paper as expected for this universe/period

In [ ]:
spy =df.filter(pl.col('ticker') == 'SPY')

spy = spy.with_columns(
    pl.col('date').dt.year().alias('year'),
    pl.col('date').dt.month().alias('month')
)

In [ ]:
monthly_spy_prices = (
    spy.sort('date')
    .group_by(['ticker', 'year', 'month'])
    .agg(pl.col('close').last().alias('price'))
    .sort(['year', 'month'])
)

In [ ]:
monthly_spy_returns = monthly_spy_prices.with_columns(
    (pl.col('price') / pl.col('price').shift(1) - 1)
    .over('ticker')
    .alias('return')
)

In [ ]:
monthly_spy_returns = monthly_spy_returns.with_columns(
    (pl.col('year').cast(pl.String) + '-' + 
     pl.col('month').cast(pl.String).str.zfill(2))
     .alias('period')
)

In [ ]:
market_returns = monthly_spy_returns.with_columns(
    (pl.col('return')
     .rolling_map(lambda x: (1 + x).product() - 1, window_size=24)).alias('rolling_24m')
)

In [ ]:
market_df = market_returns.with_columns(
    pl.when(pl.col('rolling_24m') < 0)
    .then(pl.lit(1))
    .otherwise(pl.lit(0)).alias('bear_state')
)

In [ ]:
market_df['bear_state'].value_counts()

In [ ]:
# Bear state analysis limitation:
# Only 1 bear state month identified (2023-10) in the 2011-2026 sample.
# The D&M crash mechanism requires multiple panic cycles to be statistically 
# detectable. This sample period was predominantly a bull market.
# The bear state regression cannot be meaningfully estimated on this data.
# This is a sample period limitation, not a code error.
# 
# Implication for A_6M: the regime detector will use a modified bear state
# definition — SPY 200MA + VIX flag — which is more sensitive to shorter-term
# stress periods and fires more frequently in modern markets.

In [ ]:
wml

In [ ]:
# Calculate realised volatility from WML returns
wml_with_vol = wml.with_columns(
    pl.col('wml_return')
    .rolling_std(window_size=6)
    .alias('realised_vol')
)

In [ ]:
from arch import arch_model

wml_returns = wml['wml_return'].to_numpy() * 100

model = arch_model(wml_returns, p=1, o=1, q=1, dist='normal')
result = model.fit(disp='off')
print(result.summary())

In [ ]:
winners_losers = formation_long.filter(
    pl.col('decile').is_in([1, 10])
).select(['period', 'ticker', 'decile'])

In [ ]:
winners_losers

In [ ]:
df_daily = df.with_columns([
    (pl.col('date').dt.year().cast(pl.String) + '-' +
     pl.col('date').dt.month().cast(pl.String).str.zfill(2))
    .alias('period')
])

In [ ]:
daily_wml = df_daily.join(
    winners_losers,
    on=['period', 'ticker'],
    how='inner'
)

In [ ]:
daily_wml = daily_wml.sort(['ticker', 'date']).with_columns(
    (pl.col('close') / pl.col('close').shift(1).over('ticker') - 1)
    .alias('daily_return')
)

In [ ]:
daily_wml_returns = (
    daily_wml.group_by(['date', 'decile'])
        .agg(pl.col('daily_return').mean())
        .pivot(values='daily_return', index='date', on='decile')
        .sort('date')
        .rename({'10': 'winners', '1': 'losers'})
        .with_columns(
            (pl.col('winners') - pl.col('losers')).alias('wml_daily')
        )
        .drop_nulls()
    )

In [ ]:
from arch import arch_model

wml_daily_array = daily_wml_returns['wml_daily'].to_numpy() * 100

model = arch_model(wml_daily_array, p=1, o=1, q=1, dist='normal')
result = model.fit(disp='off')
print(result.summary())

In [ ]:
model = arch_model(wml_daily_array, p=1, q=1, dist='normal')
result = model.fit(disp='off')
print(result.summary())

In [ ]:
daily_vol = result.conditional_volatility / 100

# Add to daily DataFrame
daily_wml_returns = daily_wml_returns.with_columns(
    pl.Series('garch_vol', daily_vol)
)

# Aggregate to monthly - Take the last GARCH vol of each month as the forecast
monthly_garch_vol = (
    daily_wml_returns.with_columns([
        pl.col('date').dt.year().cast(pl.String).alias('year'),
        pl.col('date').dt.month().cast(pl.String).str.zfill(2).alias('month')
    ]).with_columns(
        (pl.col('year') + '-' + pl.col('month')).alias('period')
    )
    .group_by('period')
    .agg(pl.col('garch_vol').last().alias('garch_vol'))
    .sort('period')
)

In [ ]:
wml_with_vol = wml.with_columns(
    pl.col('wml_return')
    .rolling_std(window_size=6)
    .alias('realised_vol')
).join(
    monthly_garch_vol,
    on='period'
).drop_nulls()

In [ ]:
y = wml_with_vol['realised_vol'].shift(-1).drop_nulls().to_numpy()
X = wml_with_vol.select(['garch_vol', 'realised_vol']).head(len(y)).to_numpy()

const = np.ones((len(y), 1))
X_full = np.hstack([const, X])

beta = np.linalg.lstsq(X_full, y, rcond=None)[0]
print(f"Constant:     {beta[0]:.6f}")
print(f"GARCH vol:    {beta[1]:.4f}")
print(f"Realized vol: {beta[2]:.4f}")

In [ ]:
# Combined variance forecast: GARCH component contributes negligibly 
# on this universe (coef = -0.004, effectively zero).
# Using realized volatility alone as variance forecast.
# This is consistent with the large cap universe finding — 
# volatility clustering is present (beta=0.985) but GARCH adds 
# nothing beyond what trailing realized vol already captures.

In [ ]:
variance_forecast = wml_with_vol.with_columns([
    (pl.col('realised_vol') ** 2).alias('sigma2')
]).drop_nulls()

In [ ]:
spy_variance = monthly_spy_returns.with_columns([
    (pl.col('return').rolling_std(window_size=6) ** 2)
    .alias('spy_sigma2')
]).select(['period', 'spy_sigma2']).drop_nulls()

In [ ]:
regression_data = variance_forecast.join(
    spy_variance,
    on='period'
).drop_nulls()

In [ ]:
# Create interaction regression model
X = regression_data.select('spy_sigma2').to_numpy()

const = np.ones((len(X), 1))
X_full = np.hstack([const, X.reshape(-1, 1)])

y = regression_data['wml_return'].to_numpy()

beta = np.linalg.lstsq(X_full, y, rcond=None)[0]


print(f"Constant (γ₀):    {beta[0]:.4f}")
print(f"SPY variance (γ_σ²): {beta[1]:.4f}")

In [ ]:
# Mean forecast limitation:
# Neither the bear state indicator nor SPY variance predicts WML returns
# in the expected direction on the 2011-2026 sample.
# Bear state: only 1 observation — statistically useless
# SPY variance: positive coefficient (+2.47) — opposite to paper's finding
#
# Root cause: 2011-2026 was a bull market where volatility spikes were 
# followed by recoveries, not crashes. The D&M forecasting relationship 
# requires bear market cycles to hold.
#
# Implication: the dynamic strategy will use realized vol scaling only
# (equivalent to constant volatility strategy) since the mean forecast
# cannot be estimated reliably on this sample.

In [ ]:
cvol_weights = (
    regression_data.with_columns([
        pl.col('realised_vol').shift(1).alias('vol_lag')
    ])
    .drop_nulls()
)

cvol_weights